# DAIFF U-Net forest challenger: inference

Run after `unet_train` has saved `DAIFF/unet/ckpt.pt`. Step 3 writes `unet_points.csv` (used to compare the U-Net with the Random Forest on the labelled reference sample); step 4 optionally writes full 10 m maps.


In [ ]:
# 1. Install the project and sign in. Free GPU: Runtime > Change runtime type > T4 GPU.
!git clone -q https://github.com/SamarthKale/DAIFF-Flood-Risk-Prediction.git daiff
%cd daiff
!pip install -q --no-deps -e .
!pip install -q pydantic pyyaml earthengine-api rasterio pandas
import os, numpy as np, ee
PROJECT = "YOUR-EARTH-ENGINE-PROJECT-ID"   # <- your Earth Engine project id (same one as in .env)
ee.Authenticate()                           # opens a Google sign-in; use the account that owns PROJECT
ee.Initialize(project=PROJECT)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/DAIFF"
for sub in ("chips", "unet", "unet_out"):
    os.makedirs(f"{DRIVE}/{sub}", exist_ok=True)
from daiff.config import load_config
from daiff.gee.ingest import make_ctx
cfg = load_config("configs/kopili.yaml", env={"DAIFF_GEE_PROJECT": PROJECT})
ctx = make_ctx(ee, cfg)


In [ ]:
# 2. What to predict. Do the reference points first (cheap), then the full maps only if the U-Net looks good.
import torch, pandas as pd
from daiff.forest.unet import UNet
from daiff.gee import refdata
from daiff.gee.chips import fetch_chip, training_image
from daiff.gee.forest_rf import FEATURES
YEARS = [2018, 2026]            # t0 and the latest season; add more years if the quota allows
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet(12, 2).to(dev)
model.load_state_dict(torch.load(DRIVE + "/unet/ckpt.pt", map_location=dev)["model"])
model.eval()
BANDS = tuple(FEATURES)

def prob_forest(chw):
    with torch.no_grad():
        return torch.softmax(model(torch.from_numpy(chw)[None].to(dev)), 1)[0, 1].cpu().numpy()


In [ ]:
# 3. Reference points: p(forest) at each labelling point for each year -> unet_points.csv.
from concurrent.futures import ThreadPoolExecutor
ref = pd.read_csv("reference/forest_sample_points_kopili.csv")
rows = []
for year in YEARS:
    img = training_image(ee, ctx.aoi, year, refdata.DATASETS["hansen"]).select(list(BANDS))
    def one(r):
        try:
            p = prob_forest(fetch_chip(ee, img, r.x, r.y, ctx.grid.crs, BANDS, px=64))
            return (r.point_id, year, float(p[32, 32]))
        except Exception as exc:
            print("point failed", r.point_id, exc)
            return None
    with ThreadPoolExecutor(8) as pool:
        rows += [x for x in pool.map(one, ref.itertuples()) if x]
pd.DataFrame(rows, columns=["point_id", "year", "p_forest"]).to_csv(DRIVE + "/unet_out/unet_points.csv", index=False)
print(len(rows), "predictions written to", DRIVE + "/unet_out/unet_points.csv")


In [ ]:
# 4. (Optional, heavy) full-basin class maps, one GeoTIFF per year on the 10 m grid.
# Test first with a small window: WINDOW = (row0, row1, col0, col1) in 10 m pixels, e.g. (8000, 8512, 10000, 10512).
import rasterio
from rasterio.windows import Window
WINDOW = None            # None = whole basin (about 1,400 tiles per year: run only if the points look good)
TILE, HALO = 512, 32
g10 = ctx.fine
H, W = g10.shape
r0, r1, c0, c1 = WINDOW or (0, H, 0, W)
for year in YEARS:
    path = f"/content/forest10m_{year}.tif"
    img = training_image(ee, ctx.aoi, year, refdata.DATASETS["hansen"]).select(list(BANDS))
    prof = dict(driver="GTiff", height=H, width=W, count=1, dtype="uint8", crs=g10.crs,
                transform=g10.transform, nodata=255, tiled=True, blockxsize=512, blockysize=512,
                compress="deflate")
    with rasterio.open(path, "w", **prof) as dst:
        for r in range(r0, r1, TILE):
            for c in range(c0, c1, TILE):
                h, w = min(TILE, H - r), min(TILE, W - c)
                x = g10.transform.c + (c + TILE / 2) * 10
                y = g10.transform.f - (r + TILE / 2) * 10
                try:
                    p = prob_forest(fetch_chip(ee, img, x, y, g10.crs, BANDS, px=TILE + 2 * HALO))
                except Exception as exc:
                    print("tile failed", r, c, exc)
                    continue
                cls = np.where(p[HALO:HALO + TILE, HALO:HALO + TILE] >= 0.5, 1, 0).astype("uint8")
                dst.write(cls[:h, :w], 1, window=Window(c, r, w, h))
    !cp {path} {DRIVE}/unet_out/
    print(year, "done")
# Then download My Drive/DAIFF/unet_out to the laptop and run:
#   uv run daiff forest import-unet -c configs/kopili.yaml --dir <that folder>
